# translate data type
This file include the code of different data type translate,like
- LabelMe to Yolo
- LabelMe to Vitpose(include the box crop and the coredinate translate)
- 

## creat list

In [ ]:
import json
import pandas as pd
from pathlib import Path


def generate_dataset_category_list(base_dir, target_subdirs):
    if not base_dir or not str(base_dir).strip():
        raise ValueError("Please set base_dir before running this function.")

    base_path = Path(base_dir)
    if not base_path.exists():
        raise FileNotFoundError(f"The specified base directory does not exist: {base_path}")

    dataset_records = []
    print("🔍 Begin traversing subfolders and recording file classifications...")

    for subdir_name in target_subdirs:
        subdir_path = base_path / subdir_name

        if not subdir_path.exists():
            print(f"⚠️ Skip non-existent path: {subdir_path}")
            continue

        print(f"📂 Loading categories: {subdir_name}")

        png_stems = {f.stem for f in subdir_path.glob("*.png")}
        json_stems = {f.stem for f in subdir_path.glob("*.json")}
        all_stems_in_dir = png_stems | json_stems

        for stem in sorted(list(all_stems_in_dir), key=lambda x: int(x) if x.isdigit() else x):
            dataset_records.append({
                'Filename': stem,
                'Category': subdir_name
            })

    df_output = pd.DataFrame(dataset_records)

    if not df_output.empty:
        output_file = base_path / "dataset_category_list.xlsx"
        df_output.to_excel(output_file, index=False)

        print("\n" + "=" * 50)
        print(f"🎉 Categorized list successfully created! A total of {len(df_output)} records were logged.")
        print(f"💾 File save path: {output_file.resolve()}")
        print("=" * 50)
    else:
        print("❌ Scanning complete; no image or JSON files were found in the specified folder.")


if __name__ == "__main__":
    BASE_DIRECTORY = r""  # Please set this to your actual base directory before running
    SUB_DIRECTORIES = ['cannot_used', 'normal_non_standard', 'normal_standard']
    generate_dataset_category_list(BASE_DIRECTORY, SUB_DIRECTORIES)


In [ ]:
import os
import pandas as pd

BASE_DIRECTORY = r""  # Please fill in your actual base directory before running

if not BASE_DIRECTORY or not os.path.exists(BASE_DIRECTORY):
    raise ValueError("Please set BASE_DIRECTORY to your dataset root before running this cell.")

report_path = os.path.join(BASE_DIRECTORY, "dataset_category_list.xlsx")
df_report = pd.read_excel(report_path)
report_dict = pd.Series(df_report['Category'].values, index=df_report['Filename'].astype(str)).to_dict()


## Data split

In [ ]:
import os
import shutil
import numpy as np
from sklearn.model_selection import StratifiedKFold
from collections import Counter

# 1. Set paths and class definitions
ori_data_folder = r""  # Fill in the root folder containing the original dataset before running

dst_root = r""  # Fill in the destination folder for the 5-fold split before running
keypoint_dst_root = r""  # Fill in the destination folder for the keypoint dataset before running
sub_folders = ['cannot_used', 'normal_non_standard', 'normal_standard']

if not ori_data_folder or not dst_root or not keypoint_dst_root:
    raise ValueError("Please fill in ori_data_folder, dst_root, and keypoint_dst_root before running this cell.")


def get_binary_label(sub):
    """
    Level 1 (classification task definition):
    - normal_standard: usable (1)
    - cannot_used, normal_non_standard: unusable (0)
    """
    return 1 if sub == 'normal_standard' else 0


all_samples = []

# 2. Collect data
print("🔍 Scanning the original dataset and building labels...")
for sub in sub_folders:
    folder_path = os.path.join(ori_data_folder, sub)
    if not os.path.exists(folder_path):
        continue

    images = [f for f in os.listdir(folder_path) if f.endswith('.png')]
    for img_name in images:
        name_stem = os.path.splitext(img_name)[0]
        json_path = os.path.join(folder_path, f"{name_stem}.json")

        if os.path.exists(json_path):
            all_samples.append({
                'img': os.path.join(folder_path, img_name),
                'lbl': json_path,
                'name': f"{name_stem}",
                'binary_label': get_binary_label(sub),
                'class_label': sub_folders.index(sub),
                'category_name': sub
            })

all_samples = np.array(all_samples)
stratify_labels = np.array([s['class_label'] for s in all_samples])


def print_full_stats(fold_num, train_idx, val_idx):
    """Print the full distribution statistics for both binary and three-class tasks."""
    print(f"\n" + "=" * 55)
    print(f"📊 --- Fold {fold_num} full distribution statistics ---")
    print("=" * 55)

    for mode, indices in [("Train", train_idx), ("Val", val_idx)]:
        samples = all_samples[indices]
        cls_counts = Counter([s['class_label'] for s in samples])
        bin_counts = Counter([s['binary_label'] for s in samples])
        kp_count = cls_counts[1] + cls_counts[2]

        print(f"\n📍 [{mode}] - Total samples: {len(indices)}")
        print(f"  ● Binary classification: usable (Std): {bin_counts[1]} | unusable: {bin_counts[0]}")
        print("  ● Detailed three-class distribution:")
        for i, name in enumerate(sub_folders):
            print(f"    - {name:<20}: {cls_counts[i]} samples")
        print(f"  🌟 Total keypoint samples (Std + Non-Std): {kp_count}")
        print("-" * 40)


skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for fold, (train_idx, val_idx) in enumerate(skf.split(all_samples, stratify_labels), 1):
    print_full_stats(fold, train_idx, val_idx)

    for mode, indices in [('train', train_idx), ('val', val_idx)]:
        img_dir = os.path.join(dst_root, f'fold{fold}', 'images', mode)
        lbl_dir = os.path.join(dst_root, f'fold{fold}', 'labels', mode)
        kp_img_dir = os.path.join(keypoint_dst_root, f'fold{fold}', 'images', mode)
        kp_lbl_dir = os.path.join(keypoint_dst_root, f'fold{fold}', 'labels', mode)

        os.makedirs(img_dir, exist_ok=True)
        os.makedirs(lbl_dir, exist_ok=True)

        for i in indices:
            s = all_samples[i]
            shutil.copy(s['img'], os.path.join(img_dir, f"{s['name']}.png"))
            shutil.copy(s['lbl'], os.path.join(lbl_dir, f"{s['name']}.json"))

            if s['category_name'] in ['normal_non_standard', 'normal_standard']:
                os.makedirs(kp_img_dir, exist_ok=True)
                os.makedirs(kp_lbl_dir, exist_ok=True)
                shutil.copy(s['img'], os.path.join(kp_img_dir, f"{s['name']}.png"))
                shutil.copy(s['lbl'], os.path.join(kp_lbl_dir, f"{s['name']}.json"))

print("\n✨ 5-fold split complete. The required dataset sets have been generated.")


## For ROI detection
| Model                  | Type                          |
| ---------------------- | ----------------------------- |
| YOLOv8                 | one-stage baseline            |
| YOLO26                 | one-stage SOTA                |
| Faster R-CNN           | two-stage                     |
| RT-DETR                | transformer-based detector    |

### For yolo 8, yolo 26, RT-DETR

In [ ]:
import os


def generated_yolo_file(p1, p2, img_w, img_h, file_name, category):
    if p1 is not None and p2 is not None:
        x_center = (p1[0] + p2[0]) / 2
        y_center = (p1[1] + p2[1]) / 2
        width = abs(p2[0] - p1[0])
        height = abs(p2[1] - p1[1])
        x_center /= img_w
        y_center /= img_h
        width /= img_w
        height /= img_h

        with open(file_name, 'a') as file:
            file.write(f"{category} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}\n")


In [ ]:
import os
import json
import shutil
import pandas as pd

# =========================
# Set paths and mode
# =========================
src_root = r""  # Fill in the source 5-fold dataset directory before running
mode = 1  # 1: detection only (all classes = 0), 2: detection + classification (0 or 1)

if mode == 1:
    dst_root = r""  # Fill in the detection output directory before running
else:
    dst_root = r""  # Fill in the detection + classification output directory before running

if not src_root or not dst_root:
    raise ValueError("Please set src_root and dst_root before running this cell.")

# =========================
# Traverse folds and train/val splits
# =========================
for fold_idx in range(1, 6):
    fold_name = f'fold{fold_idx}'
    for split in ['train', 'val']:
        src_label_dir = os.path.join(src_root, fold_name, 'labels', split)
        src_image_dir = os.path.join(src_root, fold_name, 'images', split)

        dst_label_dir = os.path.join(dst_root, fold_name, 'labels', split)
        dst_image_dir = os.path.join(dst_root, fold_name, 'images', split)

        os.makedirs(dst_label_dir, exist_ok=True)
        os.makedirs(dst_image_dir, exist_ok=True)

        if not os.path.exists(src_label_dir):
            continue

        for filename in os.listdir(src_label_dir):
            if filename.endswith('.json'):
                name_stem = os.path.splitext(filename)[0]
                json_path = os.path.join(src_label_dir, filename)

                category = 0
                if mode == 2:
                    category = 1 if report_dict.get(name_stem) == "normal_standard" else 0

                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)

                for shape in data.get('shapes', []):
                    if shape.get('shape_type') == 'rectangle' and 'ROI' in shape.get('label', ''):
                        p1, p2 = shape['points']
                        generated_yolo_file(p1, p2, data['imageWidth'], data['imageHeight'],
                                            os.path.join(dst_label_dir, f"{name_stem}.txt"), category)

                img_src_path = os.path.join(src_image_dir, f"{name_stem}.png")
                if os.path.exists(img_src_path):
                    shutil.copy(img_src_path, os.path.join(dst_image_dir, f"{name_stem}.png"))

print(f"✨ YOLO dataset conversion finished. Output saved at: {dst_root}")


#### creat yaml file

In [ ]:
import yaml


def create_yaml(path, train_dir, val_dir, test_dir, nc, names, yaml_file_path):
  data_config = {
      'path': path,
      'train': train_dir,
      'val': val_dir,
      'test': test_dir, #If don't have testing set, you can set it the same as val_dir
      'nc': nc,
      'names': names
  }

  with open(yaml_file_path, 'w', encoding='utf-8') as f:
      yaml.dump(data_config, f, sort_keys=False)


In [ ]:
# mode = 1  # 1: detection only (all classes are 0), 2: detection + classification (0 or 1)

mode = 2
if mode == 1:
    path = r""  # Fill in your actual base directory path before running
    nc = 1
    names = ['ROI']
else:
    path = r""  # Fill in your actual base directory path before running
    nc = 2
    names = ['cannot_used', 'normal_standard']

if not path:
    raise ValueError("Please set path before running this cell.")

for fold in range(1, 6):
    basic_path = os.path.join(path, 'fold' + str(fold))
    train_dir = 'images/train'
    val_dir = 'images/val'
    yaml_file_path = os.path.join(path, f'fold{fold}.yaml')
    create_yaml(basic_path, train_dir, val_dir, val_dir, nc, names, yaml_file_path)


### For Faster R-CNN

In [ ]:
def generated_faster_rcnn_file(p1, p2, file_name, category):
    """
    Faster R-CNN typically expects [x_min, y_min, x_max, y_max, category]
    and uses absolute pixel coordinates (not normalized).
    """
    if p1 is not None and p2 is not None:
        # Make sure to order the coordinates correctly
        x_min = min(p1[0], p2[0])
        y_min = min(p1[1], p2[1])
        x_max = max(p1[0], p2[0])
        y_max = max(p1[1], p2[1])

        #Write to the file in the required format: x_min, y_min, x_max, y_max, category
        with open(file_name, 'a') as file:
            file.write(f"{x_min:.2f} {y_min:.2f} {x_max:.2f} {y_max:.2f} {category}\n")


In [ ]:
import os
import json
import shutil
import pandas as pd

# =========================
# Set paths and mode
# =========================
src_root = r""  # Fill in the source dataset root before running
mode = 1  # 1: detection only, 2: detection + classification

if mode == 1:
    dst_root = r""  # Fill in the Faster R-CNN output directory before running
else:
    dst_root = r""  # Fill in the Faster R-CNN output directory before running

if not src_root or not dst_root:
    raise ValueError("Please set src_root and dst_root before running this cell.")

# =========================
# Run conversion
# =========================
for fold_idx in range(1, 6):
    fold_name = f'fold{fold_idx}'
    for split in ['train', 'val']:
        src_label_dir = os.path.join(src_root, fold_name, 'labels', split)
        src_image_dir = os.path.join(src_root, fold_name, 'images', split)

        dst_label_dir = os.path.join(dst_root, fold_name, 'labels', split)
        dst_image_dir = os.path.join(dst_root, fold_name, 'images', split)

        os.makedirs(dst_label_dir, exist_ok=True)
        os.makedirs(dst_image_dir, exist_ok=True)

        if not os.path.exists(src_label_dir):
            continue

        for filename in os.listdir(src_label_dir):
            if filename.endswith('.json'):
                name_stem = os.path.splitext(filename)[0]
                # Determine category based on mode and report_dict
                category = 1 if report_dict.get(name_stem) == "normal_standard" and mode == 2 else 0

                json_path = os.path.join(src_label_dir, filename)
                txt_path = os.path.join(dst_label_dir, name_stem + '.txt')

                with open(json_path, 'r', encoding='utf-8') as f:
                    data = json.load(f)
                    for shape in data.get('shapes', []):
                        if 'ROI' in shape['label'] and shape['shape_type'] == 'rectangle':
                            p1, p2 = shape['points']
                            generated_faster_rcnn_file(p1, p2, txt_path, category)

                img_src = os.path.join(src_image_dir, name_stem + '.png')
                if os.path.exists(img_src):
                    shutil.copy(img_src, os.path.join(dst_image_dir, name_stem + '.png'))

print(f"✨ Faster R-CNN dataset (images + labels) generated at: {dst_root}")


## For key point
| Model        | Type              |
| ------------ | ----------------- |
| YOLO26-pose  | one-stage         |
| ViTPose      | Transformer Pose  |

### yolo pose

In [ ]:
# data preprocessing
import json
import cv2
import os
import numpy as np
from pathlib import Path


def convert_labelme_to_yolo_pose(json_dir, img_dir, out_img_dir, out_label_dir):
    os.makedirs(out_img_dir, exist_ok=True)
    os.makedirs(out_label_dir, exist_ok=True)

    KPT_ORDER = [
        'forehead_1', 'forehead_2', 'nasion', 'palate_1', 'palate_2',
        'upper_lip', 'mandible', 'mentum_tip'
    ]

    for json_file in Path(json_dir).glob("*.json"):
        with open(json_file, 'r', encoding='utf-8') as f:
            data = json.load(f)

        img_path = Path(img_dir) / data['imagePath']
        if not img_path.exists():
            print(f"Warning: Image not found: {json_file.name}")
            continue

        img = cv2.imread(str(img_path))
        if img is None:
            print(f"Warning: Unable to read image: {img_path}")
            continue

        h_orig, w_orig = img.shape[:2]

        # Get the ROI
        roi_shapes = [s for s in data['shapes'] if s['label'] in ['able_ROI', 'ROI', 'unable_ROI']]

        if not roi_shapes:
            print(f"Skip {json_file.name}: missing ROI box")
            continue

        # Check if there are multiple ROI boxes
        if len(roi_shapes) > 1:
            print(f"❌ Error: {json_file.name} contains multiple ROI boxes. Please check annotation.")
            continue

        roi_shape = roi_shapes[0]
        rp = np.array(roi_shape['points'])
        rx1, ry1 = np.min(rp[:, 0]), np.min(rp[:, 1])
        rx2, ry2 = np.max(rp[:, 0]), np.max(rp[:, 1])

        crop_w, crop_h = rx2 - rx1, ry2 - ry1

        if crop_w <= 0 or crop_h <= 0:
            print(f"Warning: {json_file.name} has an invalid ROI box and will be skipped.")
            continue

        cropped_img = img[int(max(0, ry1)):int(min(h_orig, ry2)), int(max(0, rx1)):int(min(w_orig, rx2))]

        kpt_map = {s['label']: s['points'][0] for s in data['shapes'] if s['shape_type'] == 'point'}

        # Format: [x_center, y_center, width, height] 
        
        line = [0.5, 0.5, 1.0, 1.0]

        for name in KPT_ORDER:
            if name in kpt_map:
                px_raw = kpt_map[name][0] - rx1
                py_raw = kpt_map[name][1] - ry1

                nx = px_raw / crop_w
                ny = py_raw / crop_h

                if not (0 <= nx <= 1 and 0 <= ny <= 1):
                    print(f"⚠️ Warning: {json_file.name} keypoint {name} is outside the ROI range ({nx:.4f}, {ny:.4f})")

                nx = max(0.0, min(0.999999, nx))
                ny = max(0.0, min(0.999999, ny))
                line.extend([nx, ny, 2]) # 2 means the keypoint is visible
            else:
                line.extend([0, 0, 0]) # missing keypoint

        save_name = json_file.stem
        cv2.imwrite(os.path.join(out_img_dir, f"{save_name}.png"), cropped_img)

        with open(os.path.join(out_label_dir, f"{save_name}.txt"), 'w', encoding='utf-8') as f_out:
            formatted_coords = []
            for x in line:
                val = max(0.0, min(0.999999, abs(x)))
                formatted_coords.append(f"{val:.6f}")

            coords_str = " ".join(formatted_coords)
            line_str = f"0 {coords_str}".strip()
            f_out.write(line_str + "\n")

    print(f"✅ Conversion complete. Output directory: {out_img_dir}")


if __name__ == "__main__":
    for i in range(1, 6):
        for j in ['train', 'val']:
            convert_labelme_to_yolo_pose(
                json_dir=f'dataset/initial/5_fold_public_keypoint/fold{i}/labels/{j}',
                img_dir=f'dataset/initial/5_fold_public_keypoint/fold{i}/images/{j}',
                out_img_dir=f'dataset/Keypoint_detection/yolo_pose/fold{i}/images/{j}',
                out_label_dir=f'dataset/Keypoint_detection/yolo_pose/fold{i}/labels/{j}'
            )


### Visually inspect the label.

In [ ]:
## Visual verification of converted labels
import cv2
import os
import matplotlib.pyplot as plt


def verify_yolo_pose(img_path, txt_path):
    img = cv2.imread(img_path)
    if img is None:
        raise FileNotFoundError(f"Image file not found or unreadable: {img_path}")
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]

    # read the keypoint coordinates from the YOLO pose label file
    with open(txt_path, 'r') as f:
        line = f.readline().split()
        #Skip the first 5 elements (cls,x_c,y_c,w,h) and read the rest as keypoints
        kpts = [float(x) for x in line[5:]]

    for i in range(0, len(kpts), 3):
        nx, ny, vis = kpts[i], kpts[i + 1], kpts[i + 2]
        if vis > 0:
            px, py = int(nx * w), int(ny * h)
            cv2.circle(img, (px, py), 1, (255, 0, 0), -1)
            cv2.putText(img, str(i // 3 + 1), (px, py), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.title(f"Verify: {os.path.basename(img_path)}")
    plt.axis('off')
    plt.show()


test_fold = r""  # Fill in the directory of the converted keypoint dataset before running
if not test_fold:
    raise ValueError("Please set test_fold before running this validation cell.")

img_list = os.listdir(f'{test_fold}/images/train')
sample_img = '521.png'
verify_yolo_pose(
    f'{test_fold}/images/train/{sample_img}',
    f'{test_fold}/labels/train/{sample_img.replace(".png", ".txt")}'
)


### Create .yaml file

In [ ]:
import yaml


def create_yaml_yolo_pose(path, train_dir, val_dir, test_dir, nc, names, kpt_shape, yaml_file_path):
    """
    path: root directory of the dataset
    nc: number of classes
    names: class names
    kpt_shape: [number_of_keypoints, 3] where 3 means x, y, visible
    """
    data_config = {
        'path': path,
        'train': train_dir,
        'val': val_dir,
        'test': test_dir,
        'nc': nc,
        'names': names,
        'kpt_shape': kpt_shape, #For example, for 8 keypoints: [8, 3] which means 8 keypoints, each with x, y, visibility
    }

    with open(yaml_file_path, 'w', encoding='utf-8') as f:
        yaml.dump(data_config, f, sort_keys=False, allow_unicode=True)

    print(f"✅ Pose YAML generated: {yaml_file_path}")


In [ ]:
path = r""  # Please fill in your actual base directory before running

if not path:
    raise ValueError("Please set path before generating the pose YAML files.")

for fold in range(1, 6):
    train_dir = f'fold{fold}/images/train'
    val_dir = f'fold{fold}/images/val'
    test_dir = f'fold{fold}/images/val'
    nc = 1 #number of classes
    names = ['ROI']
    kpt_shape = [8, 3]
    yaml_file_path = f'{path}/fold{fold}.yaml'
    create_yaml_yolo_pose(path, train_dir, val_dir, test_dir, nc, names, kpt_shape, yaml_file_path)


## ViTPose

### for training data and validation data(already has the ground truth of ROI)

In [ ]:
# for training and validation
import os
import json
import cv2
import numpy as np

# =========================
# 設定與點順序定義
# =========================

# 嚴格定義點的順序 (用於 ViTPose 通道對應)
KEYPOINT_NAMES = [
    'forehead_1', 'forehead_2', 'nasion', 'palate_1', 'palate_2', 
    'upper_lip', 'mandible', 'mentum_tip'
]

def convert_to_coco(fold_dir, split, output_json,output_xlsx, dst_base):
    """Convert a single split (train/val) to COCO format."""
    coco_output = {
        "images": [],
        "annotations": [],
        "categories": [{"id": 1, "name": "fetal_profile", "keypoints": KEYPOINT_NAMES, "supercategory": "person"}]
    }
    
    img_dir = os.path.join(fold_dir, 'images', split)
    lbl_dir = os.path.join(fold_dir, 'labels', split)
    dst_img_dir = os.path.join(dst_base, os.path.basename(fold_dir), split, 'images')
    os.makedirs(dst_img_dir, exist_ok=True)
    roi_coords_log = []
    ann_id = 0
    img_id = 0

    if not os.path.exists(img_dir): return

    for img_name in os.listdir(img_dir):
        if not img_name.endswith('.png'): continue
        
        name_stem = os.path.splitext(img_name)[0]
        json_path = os.path.join(lbl_dir, f"{name_stem}.json")
        if not os.path.exists(json_path): continue

        with open(json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        # --- 第一步：找出 ROI 座標 ---
        roi_box = None
        for shape in data['shapes']:
            if shape['label'] in ['able_ROI', 'ROI', 'unable_ROI']:
                p1, p2 = shape['points']
                x1, y1 = min(p1[0], p2[0]), min(p1[1], p2[1])
                x2, y2 = max(p1[0], p2[0]), max(p1[1], p2[1])
                roi_box = (int(x1), int(y1), int(x2), int(y2))
                break
        
        if roi_box is None: continue

        # --- 第二步：裁切並儲存影像 ---
        img = cv2.imread(os.path.join(img_dir, img_name))
        #crop_img = img[roi_box[1]:roi_box[3], roi_box[0]:roi_box[2]]
        h, w = img.shape[:2]

        x1 = max(0, roi_box[0])
        y1 = max(0, roi_box[1])
        x2 = min(w, roi_box[2])
        y2 = min(h, roi_box[3])

        if x2 <= x1 or y2 <= y1:
            print(f"Invalid ROI in {img_name}")
            continue

        crop_img = img[y1:y2, x1:x2]
        cv2.imwrite(os.path.join(dst_img_dir, img_name), crop_img)
        roi_coords_log.append({
            'Image_Name': img_name,
            'x1': x1,
            'y1': y1,
            'x2': x2,
            'y2': y2
        })
        # --- 第三步：轉換關鍵點座標 ---
        # 初始化所有點為 [0, 0, 0] (x, y, visibility)
        kpts_map = {name: [0, 0, 0] for name in KEYPOINT_NAMES}
        for shape in data['shapes']:
            label = shape['label']
            if label in KEYPOINT_NAMES and shape['shape_type'] == 'point':
                px, py = shape['points'][0]
                # 轉為相對 ROI 的座標
                cw, ch = crop_img.shape[1], crop_img.shape[0]
                rel_x = px - x1
                rel_y = py - y1
                    
                if -1.0 <= rel_x <= cw + 1.0 and -1.0 <= rel_y <= ch + 1.0:
                    # 執行你寫的 Clamping 邏輯
                    final_x = max(0.0, min(float(cw - 0.000001), float(rel_x)))
                    final_y = max(0.0, min(float(ch - 0.000001), float(rel_y)))
                    
                    kpts_map[label] = [round(final_x, 2), round(final_y, 2), 2]
                else:
                    kpts_map[label] = [0, 0, 0]
                    print(f"Warning: {img_name} 的關鍵點 {label} 位於 ROI 外部，標註為不可見")

        # 扁平化關鍵點列表
        kpts_list = []
        for name in KEYPOINT_NAMES:
            kpts_list.extend(kpts_map[name])

        # --- 第四步：建構 COCO 資料結構 ---
        
        coco_output["images"].append({
            "id": ann_id,
            "file_name": img_name,
            "width": crop_img.shape[1],
            "height": crop_img.shape[0]
        })
        coco_output["annotations"].append({
            "id": ann_id,
            "image_id": img_id,
            "category_id": 1,
            "keypoints": kpts_list,
            "num_keypoints": sum(1 for i in range(2, len(kpts_list), 3) if kpts_list[i] > 0),
            "bbox": [0, 0, crop_img.shape[1], crop_img.shape[0]],
            "iscrowd": 0,
            "area": crop_img.shape[1] * crop_img.shape[0]
        })
        ann_id += 1
        img_id += 1

    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(coco_output, f, indent=4)

    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(coco_output, f, indent=4)

    # === 🟢 這裡新增：使用 Pandas 噴出全圖 ROI 絕對座標點的 Excel 對照表 ===
    if roi_coords_log:
        import pandas as pd
        df_roi_coords = pd.DataFrame(roi_coords_log)
        # 自動建立輸出資料夾防範閃退
        os.makedirs(os.path.dirname(os.path.abspath(output_xlsx)), exist_ok=True)
        df_roi_coords.to_excel(output_xlsx, index=False)
        print(f"📊 FetalClinicalEnsembleSystem 專用【ROI 像素座標點大表】已成功導出：{output_xlsx}")


In [ ]:
src_base = r""  # Fill in the source dataset root before running

dst_base = r""  # Fill in the output dataset root before running

if not src_base or not dst_base:
    raise ValueError("Please set src_base and dst_base before running this cell.")

for i in range(1, 6):
    fold_dir = os.path.join(src_base, f'fold{i}')
    for split in ['train', 'val']:
        out_json_path = os.path.join(dst_base, f'fold{i}', f'pose_{split}.json')
        out_xlsx_path = os.path.join(dst_base, f'fold{i}', f'roi_coords_{split}.xlsx')
        os.makedirs(os.path.dirname(out_json_path), exist_ok=True)
        convert_to_coco(fold_dir, split, out_json_path, out_xlsx_path, dst_base)

print(f"✨ Pose dataset and COCO labels generated at: {dst_base}")


### for testing data

In [ ]:
def reproject_and_convert_to_coco_ver2(img_dir, label_dir, new_roi_xlsx, output_json, dst_crop_img_dir=None):
    """
    Robust version: automatically resolves filename differences caused by extension case or hidden spaces.
    """
    import pandas as pd
    from pathlib import Path
    import json
    import cv2
    import os
    from tqdm import tqdm

    df_roi = pd.read_excel(new_roi_xlsx)
    df_roi['Image_Name'] = df_roi['Image_Name'].astype(str).str.strip().str.lower()
    df_roi = df_roi.drop_duplicates(subset=['Image_Name'], keep='first')
    roi_map = df_roi.set_index('Image_Name').to_dict(orient='index')

    coco_output = {
        "images": [],
        "annotations": [],
        "categories": [{"id": 1, "name": "fetal_profile", "keypoints": KEYPOINT_NAMES, "supercategory": "person"}]
    }

    if dst_crop_img_dir:
        os.makedirs(dst_crop_img_dir, exist_ok=True)

    ann_id = 0
    img_id = 0
    img_paths = sorted([p for p in Path(img_dir).iterdir() if p.suffix.lower() in ['.png', '.jpg', '.jpeg']])

    print(f"🚀 Reprojecting keypoints into the new ROI using the new Excel file... Total images: {len(img_paths)}")

    skip_by_excel_miss = 0
    skip_by_json_miss = 0

    for p in tqdm(img_paths, desc="Reprojecting Keypoints to New ROI"):
        img_file_name_lower = p.name.strip().lower()

        if img_file_name_lower not in roi_map:
            skip_by_excel_miss += 1
            continue

        orig_json_path = Path(label_dir) / f"{p.stem}.json"
        if not orig_json_path.exists():
            skip_by_json_miss += 1
            continue

        img = cv2.imread(str(p))
        if img is None:
            continue

        h_raw, w_raw = img.shape[:2]

        box_info = roi_map[img_file_name_lower]
        x1 = max(0, int(round(float(box_info['x1']))))
        y1 = max(0, int(round(float(box_info['y1']))))
        x2 = min(w_raw, int(round(float(box_info['x2']))))
        y2 = min(h_raw, int(round(float(box_info['y2']))))

        if x2 <= x1 or y2 <= y1:
            print(f"⚠️ Warning: {p.name} has an invalid ROI range in the new Excel file and will be skipped.")
            continue

        crop_img = img[y1:y2, x1:x2]
        cw, ch = crop_img.shape[1], crop_img.shape[0]

        if dst_crop_img_dir:
            cv2.imwrite(os.path.join(dst_crop_img_dir, p.name), crop_img)

        with open(orig_json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)

        kpts_map = {name: [0, 0, 0] for name in KEYPOINT_NAMES}

        for shape in data['shapes']:
            label = shape['label']
            if label in KEYPOINT_NAMES and shape['shape_type'] == 'point':
                px, py = shape['points'][0]
                rel_x = px - x1
                rel_y = py - y1

                if -1.0 <= rel_x <= cw + 1.0 and -1.0 <= rel_y <= ch + 1.0:
                    final_x = max(0.0, min(float(cw - 0.000001), float(rel_x)))
                    final_y = max(0.0, min(float(ch - 0.000001), float(rel_y)))
                    kpts_map[label] = [round(final_x, 2), round(final_y, 2), 2]
                else:
                    kpts_map[label] = [0, 0, 0]

        kpts_list = []
        for name in KEYPOINT_NAMES:
            kpts_list.extend(kpts_map[name])

        coco_output["images"].append({
            "id": img_id,
            "file_name": p.name,
            "width": cw,
            "height": ch
        })
        coco_output["annotations"].append({
            "id": ann_id,
            "image_id": img_id,
            "category_id": 1,
            "keypoints": kpts_list,
            "num_keypoints": sum(1 for i in range(2, len(kpts_list), 3) if kpts_list[i] > 0),
            "bbox": [0, 0, cw, ch],
            "iscrowd": 0,
            "area": cw * ch
        })
        ann_id += 1
        img_id += 1

    os.makedirs(os.path.dirname(os.path.abspath(output_json)), exist_ok=True)
    with open(output_json, 'w', encoding='utf-8') as f:
        json.dump(coco_output, f, indent=4)

    print(f"\n🎉 Conversion complete!")
    print(f" 🟢 Successfully written images: {img_id}")
    print(f" ❌ Skipped due to missing filename in Excel: {skip_by_excel_miss}")
    print(f" ❌ Skipped due to missing original LabelMe JSON: {skip_by_json_miss}")


In [ ]:
reproject_and_convert_to_coco_ver2(
    img_dir=r"",  # Fill in the full-image directory before running
    label_dir=r"",  # Fill in the original LabelMe JSON directory before running
    new_roi_xlsx=r"",  # Fill in the Excel file created from the new ROI before running
    output_json=r"",  # Fill in the target COCO JSON path before running
    dst_crop_img_dir=r""  # Optional: fill in the cropped-image output directory before running
)


### Visually inspect the label.

In [ ]:
import json
import cv2
import os
import random
import matplotlib.pyplot as plt


def visualize_coco_keypoints(json_path, img_dir, num_samples=5):
    # 1. Read the COCO JSON
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)

    # Build a mapping from image ID to file name
    images_map = {img['id']: img['file_name'] for img in data['images']}
    annotations = data['annotations']

    # Randomly sample a few annotations
    samples = random.sample(annotations, min(num_samples, len(annotations)))

    # Strictly define colors to distinguish different keypoints
    colors = [
        (255, 0, 0), (0, 255, 0), (0, 0, 255), (255, 255, 0),
        (255, 0, 255), (0, 255, 255), (255, 255, 255), (128, 0, 128)
    ]

    for ann in samples:
        img_name = images_map[ann['image_id']]
        img_path = os.path.join(img_dir, img_name)

        img = cv2.imread(img_path)
        if img is None:
            print(f"Image not found: {img_path}")
            continue

        # Retrieve keypoints in the format [x1, y1, v1, x2, y2, v2, ...]
        kpts = ann['keypoints']

        # Draw the bounding box (COCO format is [x, y, w, h])
        bx, by, bw, bh = ann['bbox']
        cv2.rectangle(img, (int(bx), int(by)), (int(bx + bw), int(by + bh)), (0, 255, 0), 2)

        # Draw 8 keypoints
        for i in range(0, len(kpts), 3):
            x, y, v = kpts[i], kpts[i + 1], kpts[i + 2]
            point_idx = i // 3

            if v > 0:
                cv2.circle(img, (int(x), int(y)), 1, colors[point_idx], -1)
                cv2.putText(img, str(point_idx + 1), (int(x) + 5, int(y) - 5),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, colors[point_idx], 1)

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(8, 8))
        plt.imshow(img_rgb)
        plt.title(f"Check: {img_name} (Points 1-8 order)")
        plt.axis('off')
        plt.show()


# --- Test execution ---
# Replace the path with the dataset directory you generated
base_path = r""  # Fill in the dataset directory before running
if not base_path:
    raise ValueError("Please set base_path before running this visualization cell.")

test_fold = 'fold1'
test_split = 'val'

visualize_coco_keypoints(
    json_path=os.path.join(base_path, test_fold, f'pose_{test_split}.json'),
    img_dir=os.path.join(base_path, test_fold, test_split, 'images'),
    num_samples=3
)
